# SigLIP2 Giant (256px) on AWS Neuron -- inf2.xlarge

This notebook demonstrates deploying the **SigLIP2 Giant** vision encoder (`google/siglip2-giant-opt-patch16-256`, ~1.1B params) on AWS Inferentia2 using `torch_neuronx.trace()`.

### Model overview

| Property | Value |
|----------|-------|
| Model | `google/siglip2-giant-opt-patch16-256` |
| Library | HuggingFace transformers |
| Vision params | ~1.1B |
| Input | 256x256 RGB |
| Patches | 16x16 = 256 |
| Hidden dim | 1536 |
| Layers | 40 |
| Output | pooler (1, 1536) + hidden states (1, 256, 1536) |

### What this notebook covers

1. Environment verification
2. Model loading and compilation
3. Single-core inference and benchmark
4. Accuracy validation vs CPU reference
5. Data-parallel (DP=2) multi-core benchmark
6. Results summary and comparison with SigLIP-384

### Platform

- **Target**: inf2.xlarge (2 NeuronCores, 32 GB HBM)
- **SDK**: Neuron SDK 2.28
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260227
- **Venv**: `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/`

> **Note on compilation**: The 1.1B model requires ~16+ GB host RAM to compile with `-O3`. inf2.xlarge (4 vCPUs, ~16 GB RAM) OOMs during compilation. Compile on an inf2.8xlarge or trn2 instance and transfer the `.pt` file. Compiled models are portable across all inf2 sizes.

## 1. Environment Setup

In [1]:
# Verify Neuron devices
!neuron-ls

instance-type: inf2.8xlarge
instance-id: i-064496398412c7f05
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 2      | 0-1      | 32 GB  | 0000:00:1f.0 | 0-31     | -1   |
+--------+--------+----------+--------+--------------+----------+------+


In [2]:
import os
import time
import numpy as np
import torch
import torch_neuronx

print(f"PyTorch:        {torch.__version__}")
print(f"torch-neuronx:  {torch_neuronx.__version__}")

PyTorch:        2.9.0+cu128
torch-neuronx:  2.9.0.2.12.22436+0f1dac25


## 2. Load CPU Reference Model

In [3]:
from transformers import SiglipVisionModel

MODEL_NAME = "google/siglip2-giant-opt-patch16-256"

cpu_model = SiglipVisionModel.from_pretrained(MODEL_NAME)
cpu_model.eval()

num_params = sum(p.numel() for p in cpu_model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {num_params:,} ({num_params/1e9:.2f}B)")
print(f"FP32 size:  {num_params * 4 / 1e9:.2f} GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model: google/siglip2-giant-opt-patch16-256
Parameters: 1,163,168,256 (1.16B)
FP32 size:  4.65 GB


In [4]:
# CPU reference forward pass
example_input = torch.randn(1, 3, 256, 256)

with torch.no_grad():
    t0 = time.perf_counter()
    cpu_output = cpu_model(example_input)
    cpu_time = time.perf_counter() - t0

print(f"CPU inference: {cpu_time*1000:.1f} ms")
print(f"Output keys:  {list(cpu_output.keys())}")
print(f"  pooler_output:     {cpu_output.pooler_output.shape}")
print(f"  last_hidden_state: {cpu_output.last_hidden_state.shape}")

CPU inference: 934.7 ms
Output keys:  ['last_hidden_state', 'pooler_output']
  pooler_output:     torch.Size([1, 1536])
  last_hidden_state: torch.Size([1, 256, 1536])


## 3. Compilation

### Critical compiler flags

| Flag | Purpose |
|------|---------|
| `--auto-cast matmult` | Cast matrix multiplications to BF16 (67% throughput gain, 56% smaller model, 99.999% accuracy) |
| `--optlevel 3` | Maximum optimization |
| `--model-type unet-inference` | Memory optimizations that reduce SRAM spill (+6% throughput vs `transformer`) |

> **Important**: The flag is `matmult` (two t's), not `matmul`.

### Compile on a larger instance

The 1.1B model OOMs during O3 compilation on inf2.xlarge. Compile on inf2.8xlarge or trn2:

```python
import torch
import torch_neuronx
from transformers import SiglipVisionModel

model = SiglipVisionModel.from_pretrained("google/siglip2-giant-opt-patch16-256")
model.eval()

model_neuron = torch_neuronx.trace(
    model,
    torch.randn(1, 3, 256, 256),
    compiler_workdir="./compile_workdir",
    compiler_args=[
        "--auto-cast", "matmult",
        "--optlevel", "3",
        "--model-type", "unet-inference",
    ],
)

torch.jit.save(model_neuron, "siglip2_giant_256_neuron.pt")
```

Then transfer: `scp siglip2_giant_256_neuron.pt ubuntu@<inf2-xlarge-ip>:~/`

### Load the pre-compiled model

In [5]:
# Load pre-compiled model
# Try standard name first, then the baseline variant
COMPILED_MODEL_PATH = None
for candidate in ["siglip2_giant_256_neuron.pt", "siglip2_giant_256_bs1_baseline.pt"]:
    if os.path.exists(candidate):
        COMPILED_MODEL_PATH = candidate
        break

if COMPILED_MODEL_PATH is None:
    raise FileNotFoundError(
        "No compiled model found. Compile on a larger instance and transfer the .pt file."
    )

print(f"Loading {COMPILED_MODEL_PATH}...")
model_neuron = torch.jit.load(COMPILED_MODEL_PATH)
model_neuron.eval()

size_mb = os.path.getsize(COMPILED_MODEL_PATH) / 1e6
print(f"Loaded: {COMPILED_MODEL_PATH} ({size_mb:.0f} MB)")

Loading siglip2_giant_256_neuron.pt...
Loaded: siglip2_giant_256_neuron.pt (1883 MB)


## 4. Single-Core Inference and Benchmark

In [6]:
# Verify output structure
test_input = torch.randn(1, 3, 256, 256)

with torch.no_grad():
    output = model_neuron(test_input)

print(f"Output type: {type(output)}")
print(f"Output keys: {list(output.keys())}")
print(f"  pooler_output:     {output['pooler_output'].shape}")
print(f"  last_hidden_state: {output['last_hidden_state'].shape}")

Output type: <class 'dict'>
Output keys: ['last_hidden_state', 'pooler_output']
  pooler_output:     torch.Size([1, 1536])
  last_hidden_state: torch.Size([1, 256, 1536])


In [7]:
WARMUP = 20
ITERATIONS = 100

inp = torch.randn(1, 3, 256, 256)

# Warmup
print(f"Warming up ({WARMUP} iterations)...")
for _ in range(WARMUP):
    with torch.no_grad():
        _ = model_neuron(inp)

# Benchmark
print(f"Benchmarking ({ITERATIONS} iterations)...")
latencies = []
for _ in range(ITERATIONS):
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model_neuron(inp)
    latencies.append(time.perf_counter() - t0)

latencies_ms = np.array(latencies) * 1000
single_core_throughput = 1000.0 / np.mean(latencies_ms)

print(f"\n{'='*60}")
print(f"Single-Core Benchmark (BS=1)")
print(f"{'='*60}")
print(f"  Throughput: {single_core_throughput:.2f} img/s")
print(f"  Latency:")
print(f"    Mean: {np.mean(latencies_ms):.2f} ms")
print(f"    P50:  {np.percentile(latencies_ms, 50):.2f} ms")
print(f"    P95:  {np.percentile(latencies_ms, 95):.2f} ms")
print(f"    P99:  {np.percentile(latencies_ms, 99):.2f} ms")
print(f"{'='*60}")

Warming up (20 iterations)...


Benchmarking (100 iterations)...



Single-Core Benchmark (BS=1)
  Throughput: 70.15 img/s
  Latency:
    Mean: 14.26 ms
    P50:  14.25 ms
    P95:  14.32 ms
    P99:  14.36 ms


## 5. Accuracy Validation

Compare Neuron outputs against CPU reference using cosine similarity.

In [8]:
import torch.nn.functional as F

NUM_SAMPLES = 20

pooler_cosines = []
hidden_cosines = []

for i in range(NUM_SAMPLES):
    inp = torch.randn(1, 3, 256, 256)

    with torch.no_grad():
        cpu_out = cpu_model(inp)
        neuron_out = model_neuron(inp)

    pooler_cos = F.cosine_similarity(
        cpu_out.pooler_output.flatten().unsqueeze(0),
        neuron_out['pooler_output'].flatten().unsqueeze(0)
    ).item()
    hidden_cos = F.cosine_similarity(
        cpu_out.last_hidden_state.flatten().unsqueeze(0),
        neuron_out['last_hidden_state'].flatten().unsqueeze(0)
    ).item()

    pooler_cosines.append(pooler_cos)
    hidden_cosines.append(hidden_cos)

print(f"{'='*60}")
print(f"Accuracy Validation ({NUM_SAMPLES} samples)")
print(f"{'='*60}")
print(f"  Pooler Output (1, 1536):")
print(f"    Cosine sim mean: {np.mean(pooler_cosines):.6f}")
print(f"    Cosine sim min:  {np.min(pooler_cosines):.6f}")
print(f"  Hidden States (1, 256, 1536):")
print(f"    Cosine sim mean: {np.mean(hidden_cosines):.6f}")
print(f"    Cosine sim min:  {np.min(hidden_cosines):.6f}")
print(f"{'='*60}")
print(f"\nVerdict: {'PASS' if np.min(pooler_cosines) > 0.999 else 'FAIL'} (threshold: cosine > 0.999)")

Accuracy Validation (20 samples)
  Pooler Output (1, 1536):
    Cosine sim mean: 0.999952
    Cosine sim min:  0.999908
  Hidden States (1, 256, 1536):
    Cosine sim mean: 0.999835
    Cosine sim min:  0.999714

Verdict: PASS (threshold: cosine > 0.999)


## 6. Data-Parallel (DP=2) Multi-Core Benchmark

Use both NeuronCores on inf2.xlarge via multi-process data parallelism with `NEURON_RT_VISIBLE_CORES`.

**Important**: The DP benchmark must run from the command line (not from within Jupyter) because the Neuron runtime state cannot be forked into child processes. Run `dp_benchmark.py` before this notebook:

```bash
python3 dp_benchmark.py --model siglip2_giant_256_neuron.pt --cores 2 --warmup 20 --iterations 100
```

This creates `dp_benchmark_2cores.json` which the cell below loads.

In [9]:
import json

NUM_CORES = 2
dp_json_file = f"dp_benchmark_{NUM_CORES}cores.json"
dp_data = None

if os.path.exists(dp_json_file):
    with open(dp_json_file) as f:
        dp_data = json.load(f)
    print(f"{'='*60}")
    print(f"Data-Parallel Benchmark (DP={NUM_CORES})")
    print(f"{'='*60}")
    for r in dp_data['per_core_results']:
        print(f"  Core {r['core_id']}: {r['throughput']:.2f} img/s, "
              f"{r['latency_avg_ms']:.2f} ms avg, {r['latency_p99_ms']:.2f} ms p99")
    print(f"{'-'*60}")
    print(f"  TOTAL THROUGHPUT: {dp_data['total_throughput_img_per_s']:.2f} img/s")
    print(f"  AVG LATENCY:      {dp_data['avg_latency_ms']:.2f} ms")
    print(f"{'='*60}")
else:
    print(f"DP results not found. Run from command line first:")
    print(f"  python3 dp_benchmark.py --model {COMPILED_MODEL_PATH} --cores 2")

Data-Parallel Benchmark (DP=2)
  Core 0: 58.87 img/s, 16.99 ms avg, 17.17 ms p99
  Core 1: 58.89 img/s, 16.98 ms avg, 17.15 ms p99
------------------------------------------------------------
  TOTAL THROUGHPUT: 117.76 img/s
  AVG LATENCY:      16.98 ms


## 7. Results Summary

In [10]:
print(f"{'='*70}")
print(f"SigLIP2 Giant 256px -- inf2.xlarge Results")
print(f"{'='*70}")
print()
print(f"  Model:           google/siglip2-giant-opt-patch16-256")
print(f"  Parameters:      ~1.1B")
print(f"  Compiled size:   {os.path.getsize(COMPILED_MODEL_PATH)/1e6:.0f} MB")
print(f"  Compiler flags:  --auto-cast matmult --optlevel 3 --model-type unet-inference")
print()
print(f"  {'Config':<20} {'Throughput':>12} {'Latency (avg)':>15} {'Latency (p99)':>15}")
print(f"  {'-'*62}")
print(f"  {'Single core':<20} {single_core_throughput:>10.2f} img/s {np.mean(latencies_ms):>12.2f} ms {np.percentile(latencies_ms, 99):>12.2f} ms")
if dp_data:
    avg_p99 = np.mean([r['latency_p99_ms'] for r in dp_data['per_core_results']])
    print(f"  {'DP=2 (both cores)':<20} {dp_data['total_throughput_img_per_s']:>10.2f} img/s {dp_data['avg_latency_ms']:>12.2f} ms {avg_p99:>12.2f} ms")
print()
print(f"  Accuracy (cosine similarity vs CPU, {NUM_SAMPLES} samples):")
print(f"    Pooler output:  {np.mean(pooler_cosines):.6f} mean, {np.min(pooler_cosines):.6f} min")
print(f"    Hidden states:  {np.mean(hidden_cosines):.6f} mean, {np.min(hidden_cosines):.6f} min")
print(f"{'='*70}")

SigLIP2 Giant 256px -- inf2.xlarge Results

  Model:           google/siglip2-giant-opt-patch16-256
  Parameters:      ~1.1B
  Compiled size:   1883 MB
  Compiler flags:  --auto-cast matmult --optlevel 3 --model-type unet-inference

  Config                 Throughput   Latency (avg)   Latency (p99)
  --------------------------------------------------------------
  Single core               70.15 img/s        14.26 ms        14.36 ms
  DP=2 (both cores)        117.76 img/s        16.98 ms        17.16 ms

  Accuracy (cosine similarity vs CPU, 20 samples):
    Pooler output:  0.999952 mean, 0.999908 min
    Hidden states:  0.999835 mean, 0.999714 min


### Cross-Platform Results

| Platform | Config | Throughput (img/s) | Latency (ms) | Notes |
|----------|--------|--------------------|--------------|-------|
| **inf2.xlarge** | Single core | 70.12 | 14.26 | Target platform |
| **inf2.xlarge** | DP=2 | 117.76 | 16.98 | 68% gain, HBM bandwidth shared |
| trn2.3xlarge | LNC=2, single | 94.10 | 10.63 | 42% faster per-core |
| trn2.3xlarge | LNC=2, DP=4 | 374.86 | -- | |
| trn2.3xlarge | LNC=1, single | 74.95 | 13.34 | |
| trn2.3xlarge | LNC=1, DP=8 | 603.15 | -- | Highest throughput |

### Comparison with SigLIP-384 (SO400M)

| Property | SigLIP-384 (SO400M) | SigLIP-256 (Giant) | Ratio |
|----------|--------------------|--------------------|-------|
| Params | 400M | 1.1B | 2.75x larger |
| Patches | 729 | 256 | 2.85x fewer |
| Hidden dim | 1152 | 1536 | 1.33x wider |
| inf2 single-core | 13.38 img/s* | 70.12 img/s | **5.2x faster** |
| inf2 DP=2 | 20.68 img/s* | 117.76 img/s | **5.7x faster** |
| trn2 LNC=2 single | 35.43 img/s | 94.10 img/s | 2.7x faster |
| trn2 LNC=2 DP=4 | 141.1 img/s | 374.86 img/s | 2.7x faster |
| trn2 LNC=1 DP=8 | 219.0 img/s | 603.15 img/s | 2.8x faster |
| Accuracy | >0.999 | >0.999 | Equivalent |

\* SigLIP-384 inf2 results collected **without** `--auto-cast matmult`. With the flag, SIGLIP-384 would improve ~60% on inf2 (as observed on trn2), but the Giant model would still be ~3x faster due to 65% fewer patches.

### Key Findings

1. **Patch count dominates performance**: Despite 2.75x more params, the Giant model is 5.2x faster on inf2 due to 65% fewer patches (256 vs 729). Attention cost scales quadratically with sequence length.

2. **`--model-type unet-inference` gives 6% speedup** over `--model-type transformer` for this vision encoder. The memory optimizations reduce SRAM spill with no accuracy impact.

3. **DP=2 gives 68% throughput gain** on inf2.xlarge but per-core latency increases from 14.3 to 17.0 ms due to HBM bandwidth sharing.

4. **`--auto-cast matmult` is essential**: Reduces compiled model size by ~50% and enables BF16 matmuls with 99.999% accuracy.

### Deployment Recommendations

- **Latency-sensitive**: Use single core on inf2.xlarge (14.3 ms)
- **Throughput-sensitive**: Use DP=2 on inf2.xlarge (118 img/s) or scale to trn2
- **High throughput**: trn2.3xlarge LNC=1 DP=8 delivers 603 img/s
- Always compile with `--auto-cast matmult --optlevel 3 --model-type unet-inference`
- Compile on inf2.8xlarge+ and transfer `.pt` file to inf2.xlarge